# Hyperliquid Portfolio & PnL Dashboard

Tracks account balances, open perp positions, and realized/unrealized PnL including trading fees and signed perp funding from Hyperliquid data cached in CSV. Calculates portfolio performance and risk metrics (including VaR and historical volatility), attributes PnL by asset, and compares account performance with BTC through charts and summary statistics.


In [ ]:
import hyperliquid_account_analytics_implementation as implementation

# Enter your wallet address here to fetch data for that specific wallet
WALLET_ADDRESS = "0x...c248"
WALLET_ADDRESS = "0x...07f2"


analytics = implementation.HyperliquidAnalytics(WALLET_ADDRESS)
_ = analytics.fetch_data()

## Account / PnL

Interactive account and risk summary, including account and portfolio values.

**Account value** is the latest daily all-time account-value point in the local history cache. **Portfolio value** is fetched live from Hyperliquid's portfolio endpoint. They are the same API-reported account-value measure at different freshness points, so they can differ if the cached history is behind. Neither value is the open-position notional.

In [ ]:
analytics.show_account_pnl()

## Account charts

Interactive Plotly charts show account value, daily realized and unrealized PnL, cumulative PnL, and weekly return distribution.

In [ ]:
analytics.show_account_charts()

## Account vs BTC

Shows a time-weighted account index against BTC, so deposits, withdrawals, and transfers do not move it.

In [ ]:
analytics.show_account_vs_btc()

## Traded volume vs realized PnL chart

Daily fill volume compared with realized PnL by market type.

In [ ]:
analytics.show_volume_chart()

## Open positions

Interactive sortable/filterable perp positions table, including margin type, funding and fees since open as percentages of account equity and position value, and position value as a percentage of account value. Fee ratios are available for complete USDC-denominated fee history.

In [ ]:
analytics.show_open_positions()

## Realized PnL statistics

Interactive tables report fee-net spot/perp results with signed perp funding, wins, losses, and totals. The perp output also includes liquidation count, share of closes, and liquidation PnL. Daily buckets use US Eastern time; 24H, 7D, and 30D windows are elapsed-time periods.


In [ ]:
analytics.show_realized_pnl()

## Traded volume vs PnL

Interactive table compares turnover efficiency and correlations by market type. Realized PnL / Volume (%) is realized fill PnL divided by one-sided traded notional; funding is excluded.

In [ ]:
analytics.show_volume_summary()

## PnL attribution

Interactive highlighted table breaks realized PnL, open perp uPnL, and total PnL out by asset. The Closes column shows distinct perp closes per asset, counted from available fills. Each percentage column is the signed contribution as a share of that column's gross absolute PnL, so gains are positive, losses are negative, and the colors match the dollar values. Percentages can sum to less than 100% when gains and losses offset.


In [ ]:
analytics.show_pnl_attribution()

In [ ]:
print('WALLET', WALLET_ADDRESS, 'API', implementation.API)
print('REQUESTS', implementation.requests.__file__)
print('PROXIES', implementation.requests.utils.get_environ_proxies(implementation.API))
states = implementation.fetch_perp_states(WALLET_ADDRESS)
for dex, state in states.items():
    positions = [(item.get('position', {}).get('coin'), item.get('position', {}).get('szi'), item.get('position', {}).get('unrealizedPnl')) for item in state.get('assetPositions', []) if float(item.get('position', {}).get('szi') or 0)]
    print('STATE', dex or 'default', state.get('time'), state.get('marginSummary'), positions)
orders = implementation.post({'type': 'frontendOpenOrders', 'user': WALLET_ADDRESS})
print('OPEN_ORDERS', len(orders), [(order.get('coin'), order.get('oid'), order.get('orderType')) for order in orders])